# Mapa afastado

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import contextily as ctx

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot só o mapa (sem pontos)
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

# Definir limites com base na extensão dos pontos ORTHO
xmin, ymin, xmax, ymax = gdf_ortho.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Adicionar o mapa base
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Título e formatação
#ax.set_title("Área da Barragem de Alqueva", fontsize=13)
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
from pyproj import Transformer

# ==========================================================
# FICHEIROS
# ==========================================================

asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# ==========================================================
# LER CSVs
# ==========================================================

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_file)

# ==========================================================
# MAPA PORTUGAL (sem geopandas.datasets)
# ==========================================================

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

portugal = world[world["NAME"] == "Portugal"]

# ==========================================================
# LEVEL 2 (LAT/LON)
# ==========================================================

def bbox_latlon(df):
    return box(
        df["longitude"].min(),
        df["latitude"].min(),
        df["longitude"].max(),
        df["latitude"].max()
    )

asc_poly = bbox_latlon(asc)
desc_poly = bbox_latlon(desc)

# ==========================================================
# LEVEL 3 (EPSG:3035 → WGS84)
# ==========================================================

def bbox_ortho(df):

    transformer = Transformer.from_crs(
        "EPSG:3035",
        "EPSG:4326",
        always_xy=True
    )

    minx, miny = transformer.transform(
        df["easting"].min(),
        df["northing"].min()
    )

    maxx, maxy = transformer.transform(
        df["easting"].max(),
        df["northing"].max()
    )

    return box(minx, miny, maxx, maxy)

ortho_poly = bbox_ortho(ortho)

# ==========================================================
# GeoDataFrames
# ==========================================================

asc_gdf = gpd.GeoDataFrame(
    {"name": ["Ascending LOS"]},
    geometry=[asc_poly],
    crs="EPSG:4326"
)

desc_gdf = gpd.GeoDataFrame(
    {"name": ["Descending LOS"]},
    geometry=[desc_poly],
    crs="EPSG:4326"
)

ortho_gdf = gpd.GeoDataFrame(
    {"name": ["Orthogonal Up"]},
    geometry=[ortho_poly],
    crs="EPSG:4326"
)

# ==========================================================
# PLOT
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 12))

# Portugal
portugal.plot(
    ax=ax,
    color="whitesmoke",
    edgecolor="black"
)

# Ascending
asc_gdf.plot(
    ax=ax,
    facecolor="none",
    edgecolor="blue",
    linewidth=2,
    label="Ascending LOS"
)

# Descending
desc_gdf.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    linestyle="--",
    label="Descending LOS"
)

# Orthogonal
ortho_gdf.plot(
    ax=ax,
    facecolor="none",
    edgecolor="green",
    linewidth=2,
    linestyle=":",
    label="Orthogonal Up"
)

# Extensão Portugal
ax.set_xlim(-10, -6)
ax.set_ylim(36, 43)

ax.set_title("Cobertura espacial dos produtos EGMS")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import geopandas as gpd
# import matplotlib.pyplot as plt
# from shapely.geometry import box
# import contextily as ctx
# from pyproj import Transformer

# # ==========================================================
# # FICHEIROS
# # ==========================================================

# asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

# desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# # ==========================================================
# # LER CSV
# # ==========================================================

# asc = pd.read_csv(asc_file)
# desc = pd.read_csv(desc_file)
# ortho = pd.read_csv(ortho_file)

# ==========================================================
# FUNÇÃO: BOUNDING BOX (LEVEL-2)
# ==========================================================

def bbox_latlon(df):

    return box(
        df["longitude"].min(),
        df["latitude"].min(),
        df["longitude"].max(),
        df["latitude"].max()
    )

# ==========================================================
# LEVEL-3 (easting/northing → lat/lon)
# ==========================================================

transformer = Transformer.from_crs(
    "EPSG:3035",
    "EPSG:4326",
    always_xy=True
)

def bbox_ortho(df):

    lon_min, lat_min = transformer.transform(
        df["easting"].min(),
        df["northing"].min()
    )

    lon_max, lat_max = transformer.transform(
        df["easting"].max(),
        df["northing"].max()
    )

    return box(lon_min, lat_min, lon_max, lat_max)

# ==========================================================
# POLÍGONOS
# ==========================================================

asc_poly = bbox_latlon(asc)
desc_poly = bbox_latlon(desc)
ortho_poly = bbox_ortho(ortho)

# ==========================================================
# GeoDataFrames (WGS84 → Web Mercator)
# ==========================================================

asc_gdf = gpd.GeoDataFrame(
    {"name": ["Ascending"]},
    geometry=[asc_poly],
    crs="EPSG:4326"
).to_crs(epsg=3857)

desc_gdf = gpd.GeoDataFrame(
    {"name": ["Descending"]},
    geometry=[desc_poly],
    crs="EPSG:4326"
).to_crs(epsg=3857)

ortho_gdf = gpd.GeoDataFrame(
    {"name": ["Orthogonal"]},
    geometry=[ortho_poly],
    crs="EPSG:4326"
).to_crs(epsg=3857)

# ==========================================================
# PLOT (ESTILO TESE / PUBLICAÇÃO)
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 9))

# Extensão total para zoom automático
all_bounds = pd.concat([
    asc_gdf,
    desc_gdf,
    ortho_gdf
]).total_bounds

xmin, ymin, xmax, ymax = all_bounds

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Mapa base satélite
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    alpha=0.85
)

# ==========================================================
# FOOTPRINTS
# ==========================================================

asc_gdf.boundary.plot(
    ax=ax,
    color="cyan",
    linewidth=2,
    label="Ascending LOS"
)

desc_gdf.boundary.plot(
    ax=ax,
    color="magenta",
    linewidth=2,
    linestyle="--",
    label="Descending LOS"
)

ortho_gdf.boundary.plot(
    ax=ax,
    color="yellow",
    linewidth=2,
    linestyle=":",
    label="Orthogonal Up"
)

# ==========================================================
# ESTILO FINAL
# ==========================================================

ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import geopandas as gpd
# import matplotlib.pyplot as plt
# from shapely.geometry import box
# import contextily as ctx
# from pyproj import Transformer

# # ==========================================================
# # FICHEIROS
# # ==========================================================

# asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

# desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# # ==========================================================
# # LER CSV
# # ==========================================================

# asc = pd.read_csv(asc_file)
# desc = pd.read_csv(desc_file)
# ortho = pd.read_csv(ortho_file)

# ==========================================================
# FUNÇÕES
# ==========================================================

def bbox_latlon(df):
    return box(
        df["longitude"].min(),
        df["latitude"].min(),
        df["longitude"].max(),
        df["latitude"].max()
    )

transformer = Transformer.from_crs(
    "EPSG:3035",
    "EPSG:4326",
    always_xy=True
)

def bbox_ortho(df):
    lon_min, lat_min = transformer.transform(df["easting"].min(), df["northing"].min())
    lon_max, lat_max = transformer.transform(df["easting"].max(), df["northing"].max())
    return box(lon_min, lat_min, lon_max, lat_max)

# ==========================================================
# POLÍGONOS
# ==========================================================

asc_poly = bbox_latlon(asc)
desc_poly = bbox_latlon(desc)
ortho_poly = bbox_ortho(ortho)

# ==========================================================
# GeoDataFrames → Web Mercator
# ==========================================================

asc_gdf = gpd.GeoDataFrame({"name": ["Ascending"]}, geometry=[asc_poly], crs="EPSG:4326").to_crs(3857)
desc_gdf = gpd.GeoDataFrame({"name": ["Descending"]}, geometry=[desc_poly], crs="EPSG:4326").to_crs(3857)
ortho_gdf = gpd.GeoDataFrame({"name": ["Orthogonal"]}, geometry=[ortho_poly], crs="EPSG:4326").to_crs(3857)

# ==========================================================
# PORTUGAL (MAPA BASE)
# ==========================================================

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

portugal = world[world["NAME"] == "Portugal"].to_crs(3857)

# ==========================================================
# PLOT
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 12))

# Portugal inteiro
portugal.plot(
    ax=ax,
    color="black",
    alpha=0.15,
    edgecolor="white"
)

# Basemap satélite (Portugal inteiro)
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    alpha=0.85
)

# ==========================================================
# FOOTPRINTS EGMS
# ==========================================================

asc_gdf.boundary.plot(
    ax=ax,
    color="cyan",
    linewidth=2,
    label="Ascending LOS"
)

desc_gdf.boundary.plot(
    ax=ax,
    color="magenta",
    linewidth=2,
    linestyle="--",
    label="Descending LOS"
)

ortho_gdf.boundary.plot(
    ax=ax,
    color="yellow",
    linewidth=2,
    linestyle=":",
    label="Orthogonal Up"
)

# ==========================================================
# ZOOM = PORTUGAL INTEIRO
# ==========================================================

xmin, ymin, xmax, ymax = portugal.total_bounds

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# ==========================================================
# ESTILO
# ==========================================================

ax.set_title("Cobertura espacial dos produtos EGMS sobre Portugal", fontsize=14)

ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import geopandas as gpd
# import matplotlib.pyplot as plt
# from shapely.geometry import box
# import contextily as ctx
# from pyproj import Transformer

# # ==========================================================
# # FICHEIROS
# # ==========================================================

# asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

# desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# # ==========================================================
# # LER CSV
# # ==========================================================

# asc = pd.read_csv(asc_file)
# desc = pd.read_csv(desc_file)
# ortho = pd.read_csv(ortho_file)

# ==========================================================
# LEVEL-2 (lat/lon)
# ==========================================================

def bbox_latlon(df):
    return box(
        df["longitude"].min(),
        df["latitude"].min(),
        df["longitude"].max(),
        df["latitude"].max()
    )

# ==========================================================
# LEVEL-3 (EPSG:3035 → WGS84)
# ==========================================================

transformer = Transformer.from_crs(
    "EPSG:3035",
    "EPSG:4326",
    always_xy=True
)

def bbox_ortho(df):
    lon_min, lat_min = transformer.transform(df["easting"].min(), df["northing"].min())
    lon_max, lat_max = transformer.transform(df["easting"].max(), df["northing"].max())
    return box(lon_min, lat_min, lon_max, lat_max)

# ==========================================================
# POLÍGONOS
# ==========================================================

asc_poly = bbox_latlon(asc)
desc_poly = bbox_latlon(desc)
ortho_poly = bbox_ortho(ortho)

# ==========================================================
# GeoDataFrames (Web Mercator)
# ==========================================================

asc_gdf = gpd.GeoDataFrame(
    {"name": ["Ascending"]},
    geometry=[asc_poly],
    crs="EPSG:4326"
).to_crs(3857)

desc_gdf = gpd.GeoDataFrame(
    {"name": ["Descending"]},
    geometry=[desc_poly],
    crs="EPSG:4326"
).to_crs(3857)

ortho_gdf = gpd.GeoDataFrame(
    {"name": ["Orthogonal"]},
    geometry=[ortho_poly],
    crs="EPSG:4326"
).to_crs(3857)

# ==========================================================
# PORTUGAL
# ==========================================================

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

portugal = world[world["NAME"] == "Portugal"].to_crs(3857)

# ==========================================================
# PLOT
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 12))

# Portugal base
portugal.plot(
    ax=ax,
    color="black",
    alpha=0.15,
    edgecolor="white"
)

# Basemap satélite
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldImagery,
    alpha=0.85
)

# ==========================================================
# FILL (ÁREAS SEMI-TRANSPARENTES)
# ==========================================================

asc_gdf.plot(
    ax=ax,
    facecolor="blue",
    alpha=0.25,
    edgecolor="cyan",
    linewidth=2,
    label="Ascending LOS"
)

desc_gdf.plot(
    ax=ax,
    facecolor="red",
    alpha=0.25,
    edgecolor="magenta",
    linewidth=2,
    label="Descending LOS"
)

ortho_gdf.plot(
    ax=ax,
    facecolor="green",
    alpha=0.25,
    edgecolor="yellow",
    linewidth=2,
    label="Orthogonal Up"
)

# ==========================================================
# LIMITES PORTUGAL
# ==========================================================

xmin, ymin, xmax, ymax = portugal.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# ==========================================================
# ESTILO FINAL
# ==========================================================

ax.set_title("Cobertura espacial dos produtos EGMS sobre Portugal", fontsize=14)
ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import contextily as ctx

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot apenas pontos ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

gdf_ortho.plot(ax=ax, color='yellow', edgecolor='black', markersize=30, alpha=0.8, label='Pontos ORTHO')

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Localização dos pontos ORTHO (Área da Barragem de Alqueva)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1853350, 1856050
este_min, este_max = 2791350, 2794450

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar grelha contínua e centrada nos pontos ortho
# ==============================
grid_size = 25  # metros

# Determinar o canto inferior esquerdo da grelha
x_min = ortho['easting'].min()
x_max = ortho['easting'].max()
y_min = ortho['northing'].min()
y_max = ortho['northing'].max()

# ⚙️ Ajustar limites para que o centro das células bata nos pontos ortho
# Isto é: deslocar o início da grelha meio passo para trás
x_min_aligned = x_min - grid_size / 2
y_min_aligned = y_min - grid_size / 2

# Criar limites regulares e contínuos
x_edges = np.arange(x_min_aligned, x_max + grid_size, grid_size)
y_edges = np.arange(y_min_aligned, y_max + grid_size, grid_size)

# Criar células contínuas
grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot: Grelha contínua + pontos ortho
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

grid.boundary.plot(ax=ax, color='red', linewidth=0.3, alpha=0.8, label='Grelha (100m contínua)')
#gdf_ortho.plot(ax=ax, color='white', edgecolor='black', markersize=25, alpha=0.9, label='Pontos ORTHO (centros)')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha contínua centrada nos pontos ORTHO (100m)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()

# Mapa localizado

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSV
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

ortho = ortho[
    (ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
    (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)
]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Plot apenas o mapa base (sem grelha nem pontos)
# ==============================
fig, ax = plt.subplots(figsize=(8, 6))

# Definir limites com base na extensão dos pontos ORTHO
xmin, ymin, xmax, ymax = gdf_ortho.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# Adicionar mapa base
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Título e formatação
#ax.set_title("Área selecionada sobre a Barragem de Alqueva", fontsize=13)
ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs
# ==============================
csv_ortho = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho = pd.read_csv(csv_ortho)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrame
# ==============================
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar grelha contínua e centrada nos pontos ortho
# ==============================
grid_size = 25  # metros

# Determinar o canto inferior esquerdo da grelha
x_min = ortho['easting'].min()
x_max = ortho['easting'].max()
y_min = ortho['northing'].min()
y_max = ortho['northing'].max()

# ⚙️ Ajustar limites para que o centro das células bata nos pontos ortho
# Isto é: deslocar o início da grelha meio passo para trás
x_min_aligned = x_min - grid_size / 2
y_min_aligned = y_min - grid_size / 2

# Criar limites regulares e contínuos
x_edges = np.arange(x_min_aligned, x_max + grid_size, grid_size)
y_edges = np.arange(y_min_aligned, y_max + grid_size, grid_size)

# Criar células contínuas
grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot: Grelha contínua + pontos ortho
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

grid.boundary.plot(ax=ax, color='red', linewidth=0.3, alpha=0.8, label='Grelha (100m contínua)')
#gdf_ortho.plot(ax=ax, color='white', edgecolor='black', markersize=25, alpha=0.9, label='Pontos ORTHO (centros)')
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha contínua centrada nos pontos ORTHO (100m)", fontsize=13)
ax.legend()
ax.set_axis_off()

plt.tight_layout()
plt.show()


# Pontos das órbitas ascendente e descendente

In [ ]:
# import pandas as pd
import geopandas as gpd
# import matplotlib.pyplot as plt
# from shapely.geometry import Point
import contextily as ctx

# # ==============================
# # 1. Ler CSVs ASC e DESC
# # ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrames e converter para Web Mercator
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=[Point(xy) for xy in zip(asc['easting'], asc['northing'])],
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=[Point(xy) for xy in zip(desc['easting'], desc['northing'])],
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Determinar limites comuns
# ==============================
x_min = min(gdf_asc.total_bounds[0], gdf_desc.total_bounds[0])
x_max = max(gdf_asc.total_bounds[2], gdf_desc.total_bounds[2])
y_min = min(gdf_asc.total_bounds[1], gdf_desc.total_bounds[1])
y_max = max(gdf_asc.total_bounds[3], gdf_desc.total_bounds[3])

# ==============================
# 5. Plot
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7)
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_xlim([x_min, x_max])
axes[0].set_ylim([y_min, y_max])
axes[0].set_title("Ascending")
axes[0].set_axis_off()

gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7)
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_xlim([x_min, x_max])
axes[1].set_ylim([y_min, y_max])
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

# Pontos das órbitas ascendente e descendente + pontos ortho + grelha

## Mapas separados

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Caminhos dos datasets
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ORTHO: dV e dH (atenção à troca de E/U nas pastas)
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_dh_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

# ==============================
# 2. Ler datasets
# ==============================
asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_dv = pd.read_csv(ortho_dv_file)
ortho_dh = pd.read_csv(ortho_dh_file)

# O dataset "ortho_dv" é o principal para as coordenadas
ortho = ortho_dv.copy()

# ==============================
# 3. Definir área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 50  # metros

# ==============================
# 4. Filtrar área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 5. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 6. Criar grelha contínua centrada nos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 7. Plot ASC e DESC com grelha e ORTHO
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=0.8, alpha=0.8)
gdf_ortho.plot(ax=axes[0], color='white', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (centro)')
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7, label='ASC')
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending (ASC) — grelha centrada nos ORTHO", fontsize=13)
axes[0].legend()
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=0.8, alpha=0.8)
gdf_ortho.plot(ax=axes[1], color='white', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (centro)')
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7, label='DESC')
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending (DESC) — grelha centrada nos ORTHO", fontsize=13)
axes[1].legend()
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Caminhos dos datasets
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ORTHO: dV e dH (atenção à troca de E/U nas pastas)
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_dh_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

# ==============================
# 2. Ler datasets
# ==============================
asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_dv = pd.read_csv(ortho_dv_file)
ortho_dh = pd.read_csv(ortho_dh_file)

# O dataset "ortho_dv" é o principal para as coordenadas
ortho = ortho_dv.copy()

# ==============================
# 3. Definir área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100  # metros

# ==============================
# 4. Filtrar área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 5. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 6. Criar grelha contínua centrada nos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 7. Plot ASC e DESC com grelha
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='white', linewidth=0.8, alpha=0.8)
# A linha do gdf_ortho.plot foi removida daqui
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7, label='Ascending')
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
# axes[0].set_title("Ascending (ASC) — grelha centrada nos ORTHO", fontsize=13)
axes[0].legend(fontsize=12)
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='white', linewidth=0.8, alpha=0.8)
# A linha do gdf_ortho.plot foi removida daqui
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7, label='Descending')
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
# axes[1].set_title("Descending (DESC) — grelha centrada nos ORTHO", fontsize=13)
axes[1].legend(fontsize=12)
axes[1].set_axis_off()

# Guardar o gráfico em PDF com alta resolução
plt.savefig('asc_desc_points.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Caminhos dos datasets
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

# ORTHO: dV e dH (atenção à troca de E/U nas pastas)
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_dh_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

# ==============================
# 2. Ler datasets
# ==============================
asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_dv = pd.read_csv(ortho_dv_file)
ortho_dh = pd.read_csv(ortho_dh_file)

# O dataset "ortho_dv" é o principal para as coordenadas
ortho = ortho_dv.copy()

# ==============================
# 3. Definir área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100  # metros

# ==============================
# 4. Filtrar área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 5. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 6. Criar grelha contínua centrada nos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 7. Plot ASC e DESC com grelha
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# ASCENDING
# Adicionado o label='Grid (100m)' aqui:
grid.boundary.plot(ax=axes[0], color='white', linewidth=0.8, alpha=0.8, label='Grid (100m)')
gdf_ortho.plot(ax=axes[0], color='red', edgecolor='black', markersize=30, alpha=0.9, label='Ortho')
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.7, label='Ascending')
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].legend(fontsize=12)
axes[0].set_axis_off()

# DESCENDING
# Adicionado o label='Grid (100m)' aqui:
grid.boundary.plot(ax=axes[1], color='white', linewidth=0.8, alpha=0.8, label='Grid (100m)')
gdf_ortho.plot(ax=axes[1], color='red', edgecolor='black', markersize=30, alpha=0.9, label='Ortho')
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.7, label='Descending')
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].legend(fontsize=12)
axes[1].set_axis_off()

# Ajustar o layout primeiro
plt.tight_layout()

# Guardar o gráfico em PDF com alta resolução após o ajuste de layout
plt.savefig('asc_desc_points.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.show()

## Mesmo mapa

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Definir parâmetros
# ==============================
grid_size = 100  # metros
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

# ==============================
# 2. Filtrar ASC/DESC e ORTHO na área
# ==============================
asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Criar GeoDataFrames
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(asc['easting'], asc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(desc['easting'], desc['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 4. Criar grelha contínua centrada nos pontos ORTHO
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))

grid = gpd.GeoDataFrame({'geometry': grid_polys}, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Plot
# ==============================
fig, ax = plt.subplots(figsize=(6,6))

# Grelha
grid.boundary.plot(ax=ax, color='white', linewidth=0.8, alpha=0.8, label='Grid (100m)')

# Pontos ORTHO de volta
gdf_ortho.plot(ax=ax, color='red', edgecolor='black', markersize=25, alpha=0.9, label='ORTHO (center)')

# Pontos ASC/DESC
gdf_asc.plot(ax=ax, color='blue', markersize=15, alpha=0.7, label='Ascending')
gdf_desc.plot(ax=ax, color='yellow', markersize=15, alpha=0.7, label='Descending')

# Base de mapa
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# ax.set_title("Continuous grid centered on ORTHO + ASC/DESC points", fontsize=14)
ax.legend(loc='best')
ax.set_axis_off()
plt.tight_layout()

# Guardar o gráfico em PDF com alta resolução (descomentar se necessário)
plt.savefig('combined_points.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Caminhos dos datasets
# ==============================

asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"

desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# ==============================
# 2. Ler datasets
# ==============================

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_dv_file)

# ==============================
# 3. Área de interesse
# ==============================

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

grid_size = 25

# ==============================
# 4. Filtrar área
# ==============================

def filter_area(df):

    return df[
        (df['northing'] >= norte_min) &
        (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) &
        (df['easting'] <= este_max)
    ]

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==============================
# 5. GeoDataFrames
# ==============================

gdf_asc = gpd.GeoDataFrame(
    asc,
    geometry=gpd.points_from_xy(
        asc['easting'],
        asc['northing']
    ),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc,
    geometry=gpd.points_from_xy(
        desc['easting'],
        desc['northing']
    ),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(
        ortho['easting'],
        ortho['northing']
    ),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 6. Criar grelha ORTHO
# ==============================

x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2

y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(
    x_min,
    x_max + grid_size,
    grid_size
)

y_edges = np.arange(
    y_min,
    y_max + grid_size,
    grid_size
)

grid_polys = []

for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:

        grid_polys.append(
            box(
                x0,
                y0,
                x0 + grid_size,
                y0 + grid_size
            )
        )

grid = gpd.GeoDataFrame(
    {'geometry': grid_polys},
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. LIMITES COMUNS
# ==============================

xmin, ymin, xmax, ymax = grid.total_bounds

# ==============================
# FUNÇÃO BASE
# ==============================

def setup_map(ax):

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ctx.add_basemap(
        ax,
        source=ctx.providers.Esri.WorldImagery
    )

    grid.boundary.plot(
        ax=ax,
        color='white',
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axis_off()

# ==============================
# FIGURA 1 — ORTHO
# ==============================

fig, ax = plt.subplots(figsize=(8, 8))

setup_map(ax)

gdf_ortho.plot(
    ax=ax,
    color='red',
    edgecolor='black',
    markersize=35,
    alpha=0.9,
    label='ORTHO'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'figure_ortho.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==============================
# FIGURA 2 — ASCENDING
# ==============================

fig, ax = plt.subplots(figsize=(8, 8))

setup_map(ax)

gdf_asc.plot(
    ax=ax,
    color='blue',
    markersize=20,
    alpha=0.7,
    label='Ascending'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'figure_ascending.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==============================
# FIGURA 3 — DESCENDING
# ==============================

fig, ax = plt.subplots(figsize=(8, 8))

setup_map(ax)

gdf_desc.plot(
    ax=ax,
    color='yellow',
    markersize=20,
    alpha=0.7,
    label='Descending'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'figure_descending.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==============================
# FIGURA 4 — ASC + DESC
# ==============================

fig, ax = plt.subplots(figsize=(8, 8))

setup_map(ax)

gdf_asc.plot(
    ax=ax,
    color='blue',
    markersize=20,
    alpha=0.7,
    label='Ascending'
)

gdf_desc.plot(
    ax=ax,
    color='yellow',
    markersize=20,
    alpha=0.7,
    label='Descending'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'figure_asc_desc.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

# ==============================
# FIGURA 5 — TUDO
# ==============================

fig, ax = plt.subplots(figsize=(8, 8))

setup_map(ax)

gdf_ortho.plot(
    ax=ax,
    color='red',
    edgecolor='black',
    markersize=35,
    alpha=0.9,
    label='ORTHO'
)

gdf_asc.plot(
    ax=ax,
    color='blue',
    markersize=20,
    alpha=0.7,
    label='Ascending'
)

gdf_desc.plot(
    ax=ax,
    color='yellow',
    markersize=20,
    alpha=0.7,
    label='Descending'
)

ax.legend()

plt.tight_layout()

plt.savefig(
    'figure_all.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
import random
# ==============================
# 1. Caminhos dos datasets
# ==============================
# asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
# desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
# ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# # ==============================
# # 2. Ler datasets
# # ==============================
# asc = pd.read_csv(asc_file)
# desc = pd.read_csv(desc_file)
# ortho = pd.read_csv(ortho_dv_file)

# ==============================
# 3. Área de interesse e Filtragem
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ]

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==============================
# 4. Criar Grelha Base (EPSG:3035)
# ==============================
x_min = ortho['easting'].min() - grid_size / 2
x_max = ortho['easting'].max() + grid_size / 2
y_min = ortho['northing'].min() - grid_size / 2
y_max = ortho['northing'].max() + grid_size / 2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys = []
cell_names = []
cell_idx = 0

for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))
        cell_names.append(f"Cell_{cell_idx}")
        cell_idx += 1

# Grelha original em EPSG:3035 para fazer o Join espacial correto
grid_3035 = gpd.GeoDataFrame({'cell_id': cell_names, 'geometry': grid_polys}, crs="EPSG:3035")

# ==============================
# 5. GeoDataFrames dos Pontos (EPSG:3035)
# ==============================
gdf_asc_3035 = gpd.GeoDataFrame(asc, geometry=gpd.points_from_xy(asc['easting'], asc['northing']), crs="EPSG:3035")
gdf_desc_3035 = gpd.GeoDataFrame(desc, geometry=gpd.points_from_xy(desc['easting'], desc['northing']), crs="EPSG:3035")

# Spatial Join para saber que pontos caem em que células
asc_in_cells = gpd.sjoin(gdf_asc_3035, grid_3035, how="inner", predicate="within")
desc_in_cells = gpd.sjoin(gdf_desc_3035, grid_3035, how="inner", predicate="within")

# Encontrar IDs de células que contêm dados de AMBAS as órbitas
valid_cells = list(set(asc_in_cells['cell_id']).intersection(set(desc_in_cells['cell_id'])))

if not valid_cells:
    raise ValueError("Nenhuma célula encontrada com pontos Ascending e Descending em simultâneo.")

# SELEÇÃO ALEATÓRIA DA CÉLULA
random_cell_id = random.choice(valid_cells)
print(f"Célula selecionada para o foco: {random_cell_id}")

# Isolar a geometria da célula escolhida
selected_cell_3035 = grid_3035[grid_3035['cell_id'] == random_cell_id]

# Isolar pontos que pertencem a esta célula
points_asc_cell = asc_in_cells[asc_in_cells['cell_id'] == random_cell_id]
points_desc_cell = desc_in_cells[desc_in_cells['cell_id'] == random_cell_id]

# ==============================
# 6. Reprojeção para WebMercator (EPSG:3857) para o Mapa Base
# ==============================
cell_3857 = selected_cell_3035.to_crs(epsg=3857)
gdf_asc_cell_3857 = points_asc_cell.to_crs(epsg=3857)
gdf_desc_cell_3857 = points_desc_cell.to_crs(epsg=3857)

# Obter limites da célula em 3857 para fixar o enquadramento (zoom)
xmin, ymin, xmax, ymax = cell_3857.total_bounds

# ==============================
# 7. Construção da Figura Focada
# ==============================
fig, ax = plt.subplots(figsize=(8, 8))

# Fixar os limites estritos do gráfico nas coordenadas da célula (mais uma margem de 10m para dar contexto)
padding = 10
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

# Adicionar imagem de satélite de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Desenhar o limite da célula selecionada com destaque
cell_3857.boundary.plot(
    ax=ax,
    color='red',
    linewidth=3.0,
    linestyle='-',
    label=f'Selected Cell Boundary ({random_cell_id})'
)

# Plot dos pontos Ascending detetados dentro dela
gdf_asc_cell_3857.plot(
    ax=ax,
    color='blue',
    markersize=80,
    edgecolor='white',
    alpha=0.9,
    label='Ascending PS'
)

# Plot dos pontos Descending detetados dentro dela
gdf_desc_cell_3857.plot(
    ax=ax,
    color='yellow',
    markersize=80,
    edgecolor='black',
    alpha=0.9,
    label='Descending PS'
)

# Detalhes de formatação limpa para o teu slide
ax.set_title(f"Spatial Distribution Within Single 100m Resolution Element", fontsize=12, pad=10)
ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.8)
ax.set_axis_off()

plt.tight_layout()

# Guardar o PDF focado
plt.savefig(
    'figure_random_cell_focus.pdf',
    format='pdf',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString
import contextily as ctx
import numpy as np
import random
from scipy.spatial import cKDTree

# ==========================================
# 1. Caminhos e Leitura dos Datasets
# ==========================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_dv_file)

# ==========================================
# 2. Área de Interesse e Filtragem
# ==========================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ].copy()

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==========================================
# 3. Criar Grelha Base (EPSG:3035)
# ==========================================
x_min, x_max = ortho['easting'].min() - grid_size/2, ortho['easting'].max() + grid_size/2
y_min, y_max = ortho['northing'].min() - grid_size/2, ortho['northing'].max() + grid_size/2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys, cell_names = [], []
cell_idx = 0
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))
        cell_names.append(f"Cell_{cell_idx}")
        cell_idx += 1

grid_3035 = gpd.GeoDataFrame({'cell_id': cell_names, 'geometry': grid_polys}, crs="EPSG:3035")

# ==========================================
# 4. Spatial Join dos Pontos (EPSG:3035)
# ==========================================
gdf_asc_3035 = gpd.GeoDataFrame(asc, geometry=gpd.points_from_xy(asc['easting'], asc['northing']), crs="EPSG:3035")
gdf_desc_3035 = gpd.GeoDataFrame(desc, geometry=gpd.points_from_xy(desc['easting'], desc['northing']), crs="EPSG:3035")

asc_in_cells = gpd.sjoin(gdf_asc_3035, grid_3035, how="inner", predicate="within")
desc_in_cells = gpd.sjoin(gdf_desc_3035, grid_3035, how="inner", predicate="within")

valid_cells = list(set(asc_in_cells['cell_id']).intersection(set(desc_in_cells['cell_id'])))
if not valid_cells:
    raise ValueError("Nenhuma célula partilhada com pontos ASC e DESC.")

# ==========================================
# 5. Seleção Manual ou Dinâmica da Célula
# ==========================================
selected_cell_id = "Cell_29"  # <--- Focado na célula 28, como pediste

if selected_cell_id not in valid_cells:
    print(f"Aviso: A célula {selected_cell_id} não está na lista de células válidas. Usando uma aleatória por segurança.")
    selected_cell_id = random.choice(valid_cells)

print(f"Célula selecionada para o foco: {selected_cell_id}")

selected_cell_3035 = grid_3035[grid_3035['cell_id'] == selected_cell_id]
points_asc_cell = asc_in_cells[asc_in_cells['cell_id'] == selected_cell_id]
points_desc_cell = desc_in_cells[desc_in_cells['cell_id'] == selected_cell_id]

# ==========================================
# 6. Algoritmia de Proximidade e Conexões IDW
# ==========================================
# Selecionar o primeiro ponto Ascendente como Âncora (Target s0)
anchor_pt = points_asc_cell.iloc[0]
s0_x, s0_y = anchor_pt.geometry.x, anchor_pt.geometry.y

# Encontrar os k vizinhos Descendentes mais próximos
desc_coords = np.array(list(zip(points_desc_cell.geometry.x, points_desc_cell.geometry.y)))
tree = cKDTree(desc_coords)

k_neighbors = min(5, len(points_desc_cell))
distances, indices = tree.query([s0_x, s0_y], k=k_neighbors)

# Tratar caso de k=1 para evitar erros de iteração do pandas
if k_neighbors == 1:
    indices = [indices]

neighbors_desc = points_desc_cell.iloc[indices]

# Criar as linhas tracejadas d(s0, si)
lines = [LineString([(geom.x, geom.y), (s0_x, s0_y)]) for geom in neighbors_desc.geometry]
gdf_lines_3035 = gpd.GeoDataFrame({'geometry': lines}, crs="EPSG:3035")

# Criar um raio de busca geométrico concêntrico decorativo (ex: Radius = 150m aproximado para escala gráfica)
search_radius = 50  
anchor_buffer_3035 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry.buffer(search_radius)], crs="EPSG:3035")

# ==========================================
# 7. Reprojeção para WebMercator (EPSG:3857)
# ==========================================
cell_3857 = selected_cell_3035.to_crs(epsg=3857)
anchor_3857 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry], crs="EPSG:3035").to_crs(epsg=3857)
neighbors_3857 = neighbors_desc.to_crs(epsg=3857)
gdf_lines_3857 = gdf_lines_3035.to_crs(epsg=3857)
buffer_3857 = anchor_buffer_3035.to_crs(epsg=3857)

# Definir enquadramento estrito da célula com margem de segurança (padding)
xmin, ymin, xmax, ymax = cell_3857.total_bounds
padding = 15

# ==========================================
# 8. Construção do Slide e Plotting
# ==========================================
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

# Imagem de satélite de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Desenhar limite da célula (Removida a identificação da célula)
cell_3857.boundary.plot(
    ax=ax, 
    color='#ef4444', 
    linewidth=2.5, 
    linestyle='-'#, 
    #label='Selected Cell Boundary' 
)

# Desenhar o raio de busca pontilhado
buffer_3857.boundary.plot(ax=ax, color='white', linewidth=2.5, linestyle=':', alpha=1.0, label='Search Radius')
buffer_3857.plot(ax=ax, color='white', alpha=0.05)

# Desenhar linhas de distância
gdf_lines_3857.plot(ax=ax, color='#deff9a', linewidth=2, linestyle='--', alpha=0.9, label='Euclidean Distance ($d$)')

# Desenhar vizinhos reais Descendentes (si)
neighbors_3857.plot(ax=ax, color='#eab308', markersize=140, edgecolor='black', zorder=4, label='Neighbor Descending point ($s_i$)')

# Desenhar a Âncora real Ascendente (s0)
anchor_3857.plot(ax=ax, color='#3b82f6', marker='o', markersize=220, edgecolor='white', linewidth=2, zorder=5, label='Anchor Ascending point ($s_0$)')

# ==========================================
# 9. Anotações e Caixas de Texto Estilizadas
# ==========================================
# TEXTO DA METODOLOGIA REMOVIDO (O mapa fica limpo)

# CAIXA DAS FÓRMULAS: Fundo Branco Puro, Letra Preta e Maior (fontsize=16)
props_formulas = dict(boxstyle='round,pad=0.6', facecolor='white', alpha=1.0, edgecolor='lightgray', linewidth=1)

formula_text = (
    r"$Z(s_0) = \frac{\sum_{i=1}^{k} w_i Z(s_i)}{\sum_{i=1}^{k} w_i}$"
    "\n\n"
    r"$w_i = \frac{1}{d(s_0, s_i)^p}$"
)
# A caixa das fórmulas é mantida no canto inferior esquerdo
ax.text(0.03, 0.03, formula_text, transform=ax.transAxes, fontsize=16, color='black',
        va='bottom', ha='left', bbox=props_formulas)

# Adicionar a letra "d" indicativa em cima das conexões de distância
for _, row in gdf_lines_3857.iterrows():
    coords = list(row['geometry'].coords)
    mid_x = (coords[0][0] + coords[1][0]) / 2
    mid_y = (coords[0][1] + coords[1][1]) / 2
    ax.text(mid_x, mid_y, r"$d$", color='#deff9a', fontsize=11, weight='bold', 
            bbox=dict(facecolor='#0f172a', alpha=0.6, edgecolor='none', pad=0.2))

# Ajustes finais do layout
ax.set_axis_off()
ax.legend(loc='upper right', facecolor='#0f172a', edgecolor='none', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('methodology_idw_explained_slide_clean.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString
import contextily as ctx
import numpy as np
import random
from scipy.spatial import cKDTree

# ==========================================
# 1. Caminhos e Leitura dos Datasets
# ==========================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_dv_file)

# ==========================================
# 2. Área de Interesse e Filtragem
# ==========================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ].copy()

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==========================================
# 3. Criar Grelha Base (EPSG:3035)
# ==========================================
x_min, x_max = ortho['easting'].min() - grid_size/2, ortho['easting'].max() + grid_size/2
y_min, y_max = ortho['northing'].min() - grid_size/2, ortho['northing'].max() + grid_size/2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys, cell_names = [], []
cell_idx = 0
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))
        cell_names.append(f"Cell_{cell_idx}")
        cell_idx += 1

grid_3035 = gpd.GeoDataFrame({'cell_id': cell_names, 'geometry': grid_polys}, crs="EPSG:3035")

# ==========================================
# 4. Spatial Join dos Pontos (EPSG:3035)
# ==========================================
gdf_asc_3035 = gpd.GeoDataFrame(asc, geometry=gpd.points_from_xy(asc['easting'], asc['northing']), crs="EPSG:3035")
gdf_desc_3035 = gpd.GeoDataFrame(desc, geometry=gpd.points_from_xy(desc['easting'], desc['northing']), crs="EPSG:3035")

asc_in_cells = gpd.sjoin(gdf_asc_3035, grid_3035, how="inner", predicate="within")
desc_in_cells = gpd.sjoin(gdf_desc_3035, grid_3035, how="inner", predicate="within")

valid_cells = list(set(asc_in_cells['cell_id']).intersection(set(desc_in_cells['cell_id'])))
if not valid_cells:
    raise ValueError("Nenhuma célula partilhada com pontos ASC e DESC.")

# ==========================================
# 5. Seleção Manual ou Dinâmica da Célula
# ==========================================
# Se quiseres ver os IDs disponíveis na consola, descomenta a linha abaixo:
# print("Células válidas disponíveis:", valid_cells)

selected_cell_id = "4_5"  # <--- Altera aqui o ID da célula que queres focar!

if selected_cell_id not in valid_cells:
    print(f"Aviso: A célula {selected_cell_id} não está na lista de células válidas. Usando uma aleatória por segurança.")
    selected_cell_id = random.choice(valid_cells)

print(f"Célula selecionada para o foco: {selected_cell_id}")

selected_cell_3035 = grid_3035[grid_3035['cell_id'] == selected_cell_id]
points_asc_cell = asc_in_cells[asc_in_cells['cell_id'] == selected_cell_id]
points_desc_cell = desc_in_cells[desc_in_cells['cell_id'] == selected_cell_id]

# ==========================================
# 6. Algoritmia de Proximidade e Conexões IDW
# ==========================================
# Selecionar o primeiro ponto Ascendente como Âncora (Target s0)
anchor_pt = points_asc_cell.iloc[0]
s0_x, s0_y = anchor_pt.geometry.x, anchor_pt.geometry.y

# Encontrar os k vizinhos Descendentes mais próximos
desc_coords = np.array(list(zip(points_desc_cell.geometry.x, points_desc_cell.geometry.y)))
tree = cKDTree(desc_coords)

k_neighbors = min(5, len(points_desc_cell))
distances, indices = tree.query([s0_x, s0_y], k=k_neighbors)

# Tratar caso de k=1 para evitar erros de iteração do pandas
if k_neighbors == 1:
    indices = [indices]

neighbors_desc = points_desc_cell.iloc[indices]

# Criar as linhas tracejadas d(s0, si)
lines = [LineString([(geom.x, geom.y), (s0_x, s0_y)]) for geom in neighbors_desc.geometry]
gdf_lines_3035 = gpd.GeoDataFrame({'geometry': lines}, crs="EPSG:3035")

# Criar um raio de busca geométrico concêntrico decorativo (ex: Radius = 150m aproximado para escala gráfica)
search_radius = 50  
anchor_buffer_3035 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry.buffer(search_radius)], crs="EPSG:3035")

# ==========================================
# 7. Reprojeção para WebMercator (EPSG:3857)
# ==========================================
cell_3857 = selected_cell_3035.to_crs(epsg=3857)
anchor_3857 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry], crs="EPSG:3035").to_crs(epsg=3857)
neighbors_3857 = neighbors_desc.to_crs(epsg=3857)
gdf_lines_3857 = gdf_lines_3035.to_crs(epsg=3857)
buffer_3857 = anchor_buffer_3035.to_crs(epsg=3857)

# Definir enquadramento estrito da célula com margem de segurança (padding)
xmin, ymin, xmax, ymax = cell_3857.total_bounds
padding = 15

# ==========================================
# 8. Construção do Slide e Plotting
# ==========================================
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

# Imagem de satélite de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Desenhar limite da célula (Sem o texto "100m resolution" na legenda)
cell_3857.boundary.plot(
    ax=ax, 
    color='#ef4444', 
    linewidth=2.5, 
    linestyle='-', 
    label=f'Cell {selected_cell_id}'
)

# Desenhar o raio de busca pontilhado
buffer_3857.boundary.plot(ax=ax, color='#00f5ff', linewidth=1.5, linestyle=':', alpha=0.8, label='Search Radius')
buffer_3857.plot(ax=ax, color='#00f5ff', alpha=0.05)

# Desenhar linhas de distância
gdf_lines_3857.plot(ax=ax, color='#deff9a', linewidth=2, linestyle='--', alpha=0.9, label='Euclidean Distance ($d$)')

# Desenhar vizinhos reais Descendentes (si)
neighbors_3857.plot(ax=ax, color='#eab308', markersize=140, edgecolor='black', zorder=4, label='Neighbor Descending point ($s_i$)')

# Desenhar a Âncora real Ascendente (s0)
anchor_3857.plot(ax=ax, color='#3b82f6', marker='o', markersize=220, edgecolor='white', linewidth=2, zorder=5, label='Anchor Ascending point ($s_0$)')

# ==========================================
# 9. Anotações e Caixas de Texto Estilizadas
# ==========================================
# Caixa de texto escura para a teoria à esquerda
props_text = dict(boxstyle='round,pad=0.5', facecolor='#0f172a', alpha=0.85, edgecolor='none')

explanation_text = (
    "Methodology: Spatial Synchronization\n\n"
    "1. Anchor Framework:\n"
    "   Coordinates of the Real Ascending PS\n"
    "   are fixed as the target node ($s_0$).\n\n"
    "2. IDW Inversion Parameters:\n"
    "   • Neighbors ($k = 5$): Max points collected.\n"
    "   • Power ($p = 2$): Weight decays with distance.\n"
    "   • Radius ($150m$): Spatial search constraint.\n\n"
    "3. Goal: Estimate a Virtual Descending value\n"
    "   exactly at the Ascending point coordinate."
)
ax.text(0.03, 0.97, explanation_text, transform=ax.transAxes, fontsize=10, color='white',
        va='top', ha='left', bbox=props_text, fontfamily='sans-serif')

# CAIXA DAS FÓRMULAS: Fundo Branco Puro, Letra Preta e Maior (fontsize=16)
props_formulas = dict(boxstyle='round,pad=0.6', facecolor='white', alpha=1.0, edgecolor='lightgray', linewidth=1)

formula_text = (
    r"$Z(s_0) = \frac{\sum_{i=1}^{k} w_i Z(s_i)}{\sum_{i=1}^{k} w_i}$"
    "\n\n"
    r"$w_i = \frac{1}{d(s_0, s_i)^p}$"
)
ax.text(0.03, 0.03, formula_text, transform=ax.transAxes, fontsize=16, color='black',
        va='bottom', ha='left', bbox=props_formulas)

# Adicionar a letra "d" indicativa em cima das conexões de distância
for _, row in gdf_lines_3857.iterrows():
    coords = list(row['geometry'].coords)
    mid_x = (coords[0][0] + coords[1][0]) / 2
    mid_y = (coords[0][1] + coords[1][1]) / 2
    ax.text(mid_x, mid_y, r"$d$", color='#deff9a', fontsize=11, weight='bold', 
            bbox=dict(facecolor='#0f172a', alpha=0.6, edgecolor='none', pad=0.2))

# Ajustes finais do layout
ax.set_axis_off()
ax.legend(loc='upper right', facecolor='#0f172a', edgecolor='none', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('methodology_idw_explained_slide.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString
import contextily as ctx
import numpy as np
import random
from scipy.spatial import cKDTree

# ==========================================
# 1. Caminhos e Leitura dos Datasets
# ==========================================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_dv_file)

# ==========================================
# 2. Área de Interesse e Filtragem
# ==========================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ].copy()

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==========================================
# 3. Criar Grelha Base (EPSG:3035)
# ==========================================
x_min, x_max = ortho['easting'].min() - grid_size/2, ortho['easting'].max() + grid_size/2
y_min, y_max = ortho['northing'].min() - grid_size/2, ortho['northing'].max() + grid_size/2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys, cell_names = [], []
cell_idx = 0
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))
        cell_names.append(f"Cell_{cell_idx}")
        cell_idx += 1

grid_3035 = gpd.GeoDataFrame({'cell_id': cell_names, 'geometry': grid_polys}, crs="EPSG:3035")

# ==========================================
# 4. Spatial Join dos Pontos (EPSG:3035)
# ==========================================
gdf_asc_3035 = gpd.GeoDataFrame(asc, geometry=gpd.points_from_xy(asc['easting'], asc['northing']), crs="EPSG:3035")
gdf_desc_3035 = gpd.GeoDataFrame(desc, geometry=gpd.points_from_xy(desc['easting'], desc['northing']), crs="EPSG:3035")

asc_in_cells = gpd.sjoin(gdf_asc_3035, grid_3035, how="inner", predicate="within")
desc_in_cells = gpd.sjoin(gdf_desc_3035, grid_3035, how="inner", predicate="within")

valid_cells = list(set(asc_in_cells['cell_id']).intersection(set(desc_in_cells['cell_id'])))
if not valid_cells:
    raise ValueError("Nenhuma célula partilhada com pontos ASC e DESC.")

# ==========================================
# 5. Seleção Manual ou Dinâmica da Célula
# ==========================================
selected_cell_id = "Cell_29"  # <--- Focado na célula 28, como pediste

if selected_cell_id not in valid_cells:
    print(f"Aviso: A célula {selected_cell_id} não está na lista de células válidas. Usando uma aleatória por segurança.")
    selected_cell_id = random.choice(valid_cells)

print(f"Célula selecionada para o foco: {selected_cell_id}")

selected_cell_3035 = grid_3035[grid_3035['cell_id'] == selected_cell_id]
points_asc_cell = asc_in_cells[asc_in_cells['cell_id'] == selected_cell_id]
points_desc_cell = desc_in_cells[desc_in_cells['cell_id'] == selected_cell_id]

# ==========================================
# 6. Algoritmia de Proximidade e Conexões IDW
# ==========================================
# Selecionar o primeiro ponto Ascendente como Âncora (Target s0)
anchor_pt = points_asc_cell.iloc[0]
s0_x, s0_y = anchor_pt.geometry.x, anchor_pt.geometry.y

# Encontrar os k vizinhos Descendentes mais próximos
desc_coords = np.array(list(zip(points_desc_cell.geometry.x, points_desc_cell.geometry.y)))
tree = cKDTree(desc_coords)

k_neighbors = min(5, len(points_desc_cell))
distances, indices = tree.query([s0_x, s0_y], k=k_neighbors)

# Tratar caso de k=1 para evitar erros de iteração do pandas
if k_neighbors == 1:
    indices = [indices]

neighbors_desc = points_desc_cell.iloc[indices]

# Criar as linhas tracejadas d(s0, si)
lines = [LineString([(geom.x, geom.y), (s0_x, s0_y)]) for geom in neighbors_desc.geometry]
gdf_lines_3035 = gpd.GeoDataFrame({'geometry': lines}, crs="EPSG:3035")

# Criar um raio de busca geométrico concêntrico decorativo (ex: Radius = 150m aproximado para escala gráfica)
search_radius = 50  
anchor_buffer_3035 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry.buffer(search_radius)], crs="EPSG:3035")

# ==========================================
# 7. Reprojeção para WebMercator (EPSG:3857)
# ==========================================
cell_3857 = selected_cell_3035.to_crs(epsg=3857)
anchor_3857 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry], crs="EPSG:3035").to_crs(epsg=3857)
neighbors_3857 = neighbors_desc.to_crs(epsg=3857)
gdf_lines_3857 = gdf_lines_3035.to_crs(epsg=3857)
buffer_3857 = anchor_buffer_3035.to_crs(epsg=3857)

# Definir enquadramento estrito da célula com margem de segurança (padding)
xmin, ymin, xmax, ymax = cell_3857.total_bounds
padding = 15

# ==========================================
# 8. Construção do Slide e Plotting
# ==========================================
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

# Imagem de satélite de fundo
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

# Desenhar limite da célula (Removida a identificação da célula)
cell_3857.boundary.plot(
    ax=ax, 
    color='#ef4444', 
    linewidth=2.5, 
    linestyle='-', 
    label='Selected Cell Boundary' 
)

# Desenhar o raio de busca pontilhado
buffer_3857.boundary.plot(ax=ax, color='#00f5ff', linewidth=1.5, linestyle=':', alpha=0.8, label='Search Radius')
buffer_3857.plot(ax=ax, color='#00f5ff', alpha=0.05)

# Desenhar linhas de distância
gdf_lines_3857.plot(ax=ax, color='#deff9a', linewidth=2, linestyle='--', alpha=0.9, label='Euclidean Distance ($d$)')

# Desenhar vizinhos reais Descendentes (si)
neighbors_3857.plot(ax=ax, color='#eab308', markersize=140, edgecolor='black', zorder=4, label='Neighbor Descending point ($s_i$)')

# Desenhar a Âncora real Ascendente (s0)
anchor_3857.plot(ax=ax, color='#3b82f6', marker='o', markersize=220, edgecolor='white', linewidth=2, zorder=5, label='Anchor Ascending point ($s_0$)')

# ==========================================
# 9. Anotações e Caixas de Texto Estilizadas
# ==========================================
# TEXTO DA METODOLOGIA REMOVIDO (O mapa fica limpo)

# CAIXA DAS FÓRMULAS: Fundo Branco Puro, Letra Preta e Maior (fontsize=16)
props_formulas = dict(boxstyle='round,pad=0.6', facecolor='white', alpha=1.0, edgecolor='lightgray', linewidth=1)

formula_text = (
    r"$Z(s_0) = \frac{\sum_{i=1}^{k} w_i Z(s_i)}{\sum_{i=1}^{k} w_i}$"
    "\n\n"
    r"$w_i = \frac{1}{d(s_0, s_i)^p}$"
)
# A caixa das fórmulas é mantida no canto inferior esquerdo
ax.text(0.03, 0.03, formula_text, transform=ax.transAxes, fontsize=16, color='black',
        va='bottom', ha='left', bbox=props_formulas)

# Adicionar a letra "d" indicativa em cima das conexões de distância
for _, row in gdf_lines_3857.iterrows():
    coords = list(row['geometry'].coords)
    mid_x = (coords[0][0] + coords[1][0]) / 2
    mid_y = (coords[0][1] + coords[1][1]) / 2
    ax.text(mid_x, mid_y, r"$d$", color='#deff9a', fontsize=11, weight='bold', 
            bbox=dict(facecolor='#0f172a', alpha=0.6, edgecolor='none', pad=0.2))

# Ajustes finais do layout
ax.set_axis_off()
ax.legend(loc='upper right', facecolor='#0f172a', edgecolor='none', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('methodology_idw_explained_slide_clean.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, LineString
import contextily as ctx
import numpy as np
import random
from scipy.spatial import cKDTree

# ==========================================
# 1. Caminhos e Leitura dos Datasets
# ==========================================
# asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
# desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
# ortho_dv_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

# asc = pd.read_csv(asc_file)
# desc = pd.read_csv(desc_file)
# ortho = pd.read_csv(ortho_dv_file)

# ==========================================
# 2. Área de Interesse e Filtragem
# ==========================================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250
grid_size = 100

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting'] >= este_min) & (df['easting'] <= este_max)
    ].copy()

asc = filter_area(asc)
desc = filter_area(desc)
ortho = filter_area(ortho)

# ==========================================
# 3. Criar Grelha Base (EPSG:3035)
# ==========================================
x_min, x_max = ortho['easting'].min() - grid_size/2, ortho['easting'].max() + grid_size/2
y_min, y_max = ortho['northing'].min() - grid_size/2, ortho['northing'].max() + grid_size/2

x_edges = np.arange(x_min, x_max + grid_size, grid_size)
y_edges = np.arange(y_min, y_max + grid_size, grid_size)

grid_polys, cell_names = [], []
cell_idx = 0
for x0 in x_edges[:-1]:
    for y0 in y_edges[:-1]:
        grid_polys.append(box(x0, y0, x0 + grid_size, y0 + grid_size))
        cell_names.append(f"Cell_{cell_idx}")
        cell_idx += 1

grid_3035 = gpd.GeoDataFrame({'cell_id': cell_names, 'geometry': grid_polys}, crs="EPSG:3035")

# ==========================================
# 4. Spatial Join dos Pontos (EPSG:3035)
# ==========================================
gdf_asc_3035 = gpd.GeoDataFrame(asc, geometry=gpd.points_from_xy(asc['easting'], asc['northing']), crs="EPSG:3035")
gdf_desc_3035 = gpd.GeoDataFrame(desc, geometry=gpd.points_from_xy(desc['easting'], desc['northing']), crs="EPSG:3035")

asc_in_cells = gpd.sjoin(gdf_asc_3035, grid_3035, how="inner", predicate="within")
desc_in_cells = gpd.sjoin(gdf_desc_3035, grid_3035, how="inner", predicate="within")

valid_cells = list(set(asc_in_cells['cell_id']).intersection(set(desc_in_cells['cell_id'])))
if not valid_cells:
    raise ValueError("Nenhuma célula partilhada com pontos ASC e DESC.")

# ==========================================
# 5. Seleção da Célula
# ==========================================
selected_cell_id = "4_5"  # <--- Altera aqui o ID da célula que queres focar!

if selected_cell_id not in valid_cells:
    selected_cell_id = random.choice(valid_cells)

print(f"Célula selecionada para o foco: {selected_cell_id}")

selected_cell_3035 = grid_3035[grid_3035['cell_id'] == selected_cell_id]
points_asc_cell = asc_in_cells[asc_in_cells['cell_id'] == selected_cell_id]
points_desc_cell = desc_in_cells[desc_in_cells['cell_id'] == selected_cell_id]

# ==========================================
# 6. Algoritmia de Proximidade e Conexões IDW
# ==========================================
anchor_pt = points_asc_cell.iloc[0]
s0_x, s0_y = anchor_pt.geometry.x, anchor_pt.geometry.y

desc_coords = np.array(list(zip(points_desc_cell.geometry.x, points_desc_cell.geometry.y)))
tree = cKDTree(desc_coords)

k_neighbors = min(5, len(points_desc_cell))
distances, indices = tree.query([s0_x, s0_y], k=k_neighbors)

if k_neighbors == 1:
    indices = [indices]

neighbors_desc = points_desc_cell.iloc[indices]

lines = [LineString([(geom.x, geom.y), (s0_x, s0_y)]) for geom in neighbors_desc.geometry]
gdf_lines_3035 = gpd.GeoDataFrame({'geometry': lines}, crs="EPSG:3035")

search_radius = 50  
anchor_buffer_3035 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry.buffer(search_radius)], crs="EPSG:3035")

# ==========================================
# 7. Reprojeção para WebMercator (EPSG:3857)
# ==========================================
cell_3857 = selected_cell_3035.to_crs(epsg=3857)
anchor_3857 = gpd.GeoDataFrame(geometry=[anchor_pt.geometry], crs="EPSG:3035").to_crs(epsg=3857)
neighbors_3857 = neighbors_desc.to_crs(epsg=3857)
gdf_lines_3857 = gdf_lines_3035.to_crs(epsg=3857)
buffer_3857 = anchor_buffer_3035.to_crs(epsg=3857)

xmin, ymin, xmax, ymax = cell_3857.total_bounds
padding = 15

# ==========================================
# 8. Construção do Slide e Plotting
# ==========================================
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(xmin - padding, xmax + padding)
ax.set_ylim(ymin - padding, ymax + padding)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)

cell_3857.boundary.plot(
    ax=ax, 
    color='#ef4444', 
    linewidth=2.5, 
    linestyle='-', 
    label=f'Cell {selected_cell_id}'
)

buffer_3857.boundary.plot(ax=ax, color='#00f5ff', linewidth=1.5, linestyle=':', alpha=0.8, label='Search Radius (m)')
buffer_3857.plot(ax=ax, color='#00f5ff', alpha=0.05)

gdf_lines_3857.plot(ax=ax, color='#deff9a', linewidth=2, linestyle='--', alpha=0.9, label='Distance ($d$)')

neighbors_3857.plot(ax=ax, color='#eab308', markersize=140, edgecolor='black', zorder=4, label='Neighbor Descending point ($s_i$)')

anchor_3857.plot(ax=ax, color='#3b82f6', marker='o', markersize=220, edgecolor='white', linewidth=2, zorder=5, label='Anchor Ascending point ($s_0$)')

# ==========================================
# 9. Anotações e Legenda de Termos Simples
# ==========================================
# Caixa de texto escura descritiva à esquerda
props_text = dict(boxstyle='round,pad=0.5', facecolor='#0f172a', alpha=0.85, edgecolor='none')

explanation_text = (
    "Methodology: Spatial Synchronization\n\n"
    "1. Anchor Framework:\n"
    "   Coordinates of the Real Ascending PS\n"
    "   are fixed as the target node ($s_0$).\n\n"
    "2. IDW Inversion Parameters:\n"
    "   • Neighbors ($k = 5$): Max points collected.\n"
    "   • Power ($p = 2$): Weight decays with distance.\n"
    "   • Radius ($150m$): Spatial search constraint.\n\n"
    "3. Goal: Estimate a Virtual Descending value\n"
    "   exactly at the Ascending point coordinate."
)
ax.text(0.03, 0.97, explanation_text, transform=ax.transAxes, fontsize=10, color='white',
        va='top', ha='left', bbox=props_text, fontfamily='sans-serif')

# NOVA LEGENDA SIMPLIFICADA (Fundo Branco, Sem fórmulas complexas)
props_legend = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=1.0, edgecolor='lightgray', linewidth=1)

legend_text = (
    "IDW Notation Guide:\n"
    "• $s_0$: Target Anchor Node\n"
    "• $s_i$: Known Neighboring Points\n"
    "• $Z(s_0)$: Estimated Virtual Value\n"
    "• $Z(s_i)$: Known Measured Values\n"
    "• $w_i$: Spatial Weight Factor"
)
ax.text(0.03, 0.03, legend_text, transform=ax.transAxes, fontsize=11, color='black',
        va='bottom', ha='left', bbox=props_legend, fontfamily='sans-serif')

# Indicação física "d" sobre os alinhamentos
for _, row in gdf_lines_3857.iterrows():
    coords = list(row['geometry'].coords)
    mid_x = (coords[0][0] + coords[1][0]) / 2
    mid_y = (coords[0][1] + coords[1][1]) / 2
    ax.text(mid_x, mid_y, r"$d$", color='#deff9a', fontsize=11, weight='bold', 
            bbox=dict(facecolor='#0f172a', alpha=0.6, edgecolor='none', pad=0.2))

ax.set_axis_off()
ax.legend(loc='upper right', facecolor='#0f172a', edgecolor='none', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('methodology_idw_explained_slide.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

# Mapa com ID das células

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Definir datas comuns e interpolar
# ==============================
def melt_to_long(df):
    """Transforma colunas de datas em formato longo"""
    disp_cols = df.columns[24:]  # ajuste se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Definir grelha fixa
# ==============================
grid_size = 25  # metros

xmin = min(asc_interp['easting'].min(), desc_interp['easting'].min())
xmax = max(asc_interp['easting'].max(), desc_interp['easting'].max())
ymin = min(asc_interp['northing'].min(), desc_interp['northing'].min())
ymax = max(asc_interp['northing'].max(), desc_interp['northing'].max())

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

# ==============================
# 5. Função para pontos médios por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    # Adicionar cell_id
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. GeoDataFrames para plot
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc_cells,
    geometry=gpd.points_from_xy(asc_cells['x_center'], asc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc_cells,
    geometry=gpd.points_from_xy(desc_cells['x_center'], desc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Criar grelha como polígonos com IDs
# ==============================
grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0+grid_size, y0+grid_size)
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 8. Plot com IDs
# ==============================
margin = grid_size * 0.05

fig, axes = plt.subplots(1, 2, figsize=(16,8))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=1, alpha=0.8)
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[0].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending")
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=1, alpha=0.8)
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[1].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

# ==============================
# 3. Definir datas comuns e interpolar
# ==============================
def melt_to_long(df):
    """Transforma colunas de datas em formato longo"""
    disp_cols = df.columns[24:]  # ajuste se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Definir grelha fixa
# ==============================
grid_size = 100  # metros

xmin = min(asc_interp['easting'].min(), desc_interp['easting'].min())
xmax = max(asc_interp['easting'].max(), desc_interp['easting'].max())
ymin = min(asc_interp['northing'].min(), desc_interp['northing'].min())
ymax = max(asc_interp['northing'].max(), desc_interp['northing'].max())

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

# ==============================
# 5. Função para pontos médios por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    # Adicionar cell_id
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. GeoDataFrames para plot
# ==============================
gdf_asc = gpd.GeoDataFrame(
    asc_cells,
    geometry=gpd.points_from_xy(asc_cells['x_center'], asc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

gdf_desc = gpd.GeoDataFrame(
    desc_cells,
    geometry=gpd.points_from_xy(desc_cells['x_center'], desc_cells['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Criar grelha como polígonos com IDs
# ==============================
grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0+grid_size, y0+grid_size)
        })

grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 8. Plot com IDs
# ==============================
margin = grid_size * 0.05

fig, axes = plt.subplots(1, 2, figsize=(16,8))

# ASCENDING
grid.boundary.plot(ax=axes[0], color='red', linewidth=1, alpha=0.8)
gdf_asc.plot(ax=axes[0], color='blue', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[0].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[0], source=ctx.providers.Esri.WorldImagery)
axes[0].set_title("Ascending")
axes[0].set_axis_off()

# DESCENDING
grid.boundary.plot(ax=axes[1], color='red', linewidth=1, alpha=0.8)
gdf_desc.plot(ax=axes[1], color='yellow', markersize=20, alpha=0.9)
for _, row in grid.iterrows():
    minx, miny, maxx, maxy = row['geometry'].bounds
    axes[1].text(maxx - margin, maxy - margin, row['cell_id'],
                 fontsize=6, ha='right', va='top', color='black',
                 bbox=dict(facecolor='white', alpha=0.7, pad=0.3))
ctx.add_basemap(axes[1], source=ctx.providers.Esri.WorldImagery)
axes[1].set_title("Descending")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Interpolação temporal ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Criar grelha centrada nos pontos ORTHO
# ==============================
grid_size = 50  # metros

xmin = ortho['easting'].min() - grid_size/2
xmax = ortho['easting'].max() + grid_size/2
ymin = ortho['northing'].min() - grid_size/2
ymax = ortho['northing'].max() + grid_size/2

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0 + grid_size, y0 + grid_size)
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Função para média por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. Converter para GeoDataFrames
# ==============================
def create_gdf(df):
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['x_center'], df['y_center']),
        crs="EPSG:3035"
    ).to_crs(epsg=3857)

gdf_asc = create_gdf(asc_cells)
gdf_desc = create_gdf(desc_cells)
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Plot com IDs e alinhamento ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(12,12))

# Grelha centrada nos ORTHO
grid.boundary.plot(ax=ax, color='red', linewidth=1, alpha=0.8, label=f'Grelha Base ({grid_size} m)')

# Pontos ORTHO (centro de referência)
gdf_ortho.plot(ax=ax, color='black', markersize=25, label='Pontos ORTHO')

# Pontos ASC/DESC (sobrepostos)
gdf_asc.plot(ax=ax, color='white', edgecolor='black', markersize=35, alpha=0.9, label='ASC')
gdf_desc.plot(ax=ax, color='yellow', markersize=25, alpha=0.8, label='DESC')

# IDs das células
for _, row in grid.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_max - grid_size*0.05, y_max - grid_size*0.05, row['cell_id'],
        fontsize=6, ha='right', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=0.3)
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha centrada nos ORTHO com pontos ASC/DESC", fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np

# ==============================
# 1. Ler CSVs ASC e DESC
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho = pd.read_csv(ortho_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]

desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]

ortho = ortho[(ortho['northing'] >= norte_min) & (ortho['northing'] <= norte_max) &
              (ortho['easting'] >= este_min) & (ortho['easting'] <= este_max)]

# ==============================
# 3. Interpolação temporal ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar se necessário
    long_df = df.melt(id_vars=['easting','northing'], 
                      value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'date': common_dates,
            'disp': interp
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 4. Criar grelha centrada nos pontos ORTHO
# ==============================
grid_size = 100  # metros

xmin = ortho['easting'].min() - grid_size/2
xmax = ortho['easting'].max() + grid_size/2
ymin = ortho['northing'].min() - grid_size/2
ymax = ortho['northing'].max() + grid_size/2

x_edges = np.arange(xmin, xmax + grid_size, grid_size)
y_edges = np.arange(ymin, ymax + grid_size, grid_size)

grid_data = []
for ix, x0 in enumerate(x_edges[:-1]):
    for iy, y0 in enumerate(y_edges[:-1]):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x0, y0, x0 + grid_size, y0 + grid_size)
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 5. Função para média por célula
# ==============================
def grid_average(df, grid_size, x_edges, y_edges):
    df['cell_x'] = pd.cut(df['easting'], bins=x_edges, labels=False).astype('Int64')
    df['cell_y'] = pd.cut(df['northing'], bins=y_edges, labels=False).astype('Int64')
    
    valid = df.dropna(subset=['cell_x','cell_y']).copy()
    valid['x_center'] = x_edges[valid['cell_x'].to_numpy()] + grid_size/2
    valid['y_center'] = y_edges[valid['cell_y'].to_numpy()] + grid_size/2
    
    grouped = valid.groupby(['cell_x','cell_y','date']).agg(
        x_center=('x_center','first'),
        y_center=('y_center','first'),
        disp=('disp','mean')
    ).reset_index()
    
    grouped['cell_id'] = grouped['cell_x'].astype(str) + "_" + grouped['cell_y'].astype(str)
    return grouped

asc_cells = grid_average(asc_interp, grid_size, x_edges, y_edges)
desc_cells = grid_average(desc_interp, grid_size, x_edges, y_edges)

# ==============================
# 6. Converter para GeoDataFrames
# ==============================
def create_gdf(df):
    return gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['x_center'], df['y_center']),
        crs="EPSG:3035"
    ).to_crs(epsg=3857)

gdf_asc = create_gdf(asc_cells)
gdf_desc = create_gdf(desc_cells)
gdf_ortho = gpd.GeoDataFrame(
    ortho,
    geometry=gpd.points_from_xy(ortho['easting'], ortho['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Plot com IDs e alinhamento ORTHO
# ==============================
fig, ax = plt.subplots(figsize=(12,12))

# Grelha centrada nos ORTHO
grid.boundary.plot(ax=ax, color='red', linewidth=1, alpha=0.8, label=f'Grelha Base ({grid_size} m)')

# Pontos ORTHO (centro de referência)
gdf_ortho.plot(ax=ax, color='black', markersize=25, label='Pontos ORTHO')

# Pontos ASC/DESC (sobrepostos)
gdf_asc.plot(ax=ax, color='white', edgecolor='black', markersize=35, alpha=0.9, label='ASC')
gdf_desc.plot(ax=ax, color='yellow', markersize=25, alpha=0.8, label='DESC')

# IDs das células
for _, row in grid.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_max - grid_size*0.05, y_max - grid_size*0.05, row['cell_id'],
        fontsize=6, ha='right', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=0.3)
    )

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Grelha centrada nos ORTHO com pontos ASC/DESC", fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filtrar_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filtrar_area(asc)
desc = filtrar_area(desc)
ortho_v = filtrar_area(ortho_v)
ortho_h = filtrar_area(ortho_h)

# ==============================
# 3. Criar grelha base (100 m)
# ==============================
grid_size = 100
x_edges = np.arange(asc['easting'].min(), asc['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc['northing'].min(), asc['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

# Atribuir células ASC
asc['cell_x'] = pd.cut(asc['easting'], bins=x_edges_shifted, labels=False)
asc['cell_y'] = pd.cut(asc['northing'], bins=y_edges_shifted, labels=False)

# Calcular centroide de cada célula com base nos pontos ASC
agg = asc.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 4. Criar GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

points = gpd.GeoDataFrame(
    agg,
    geometry=gpd.points_from_xy(agg['x_center'], agg['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 5. Atribuir células a ORTHO
# ==============================
for ortho_df in [ortho_v, ortho_h]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

# ==============================
# 6. Células selecionadas
# ==============================
selected_ids = [
    "2_5","3_5","4_5","5_5","6_5","7_5",
    "2_4","3_4","4_4","5_4","6_4","7_4",
    "2_6","3_6","4_6","5_6","6_6","7_6"
]

agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = points[points['cell_id'].isin(selected_ids)]

ortho_v_sel = ortho_v[ortho_v['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h[ortho_h['cell_id'].isin(selected_ids)]

# Combinar ORTHO
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing'])
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 7. Mapa Final
# ==============================
fig, ax = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax, color='lightgray', linewidth=0.5, label='Grelha Base (100 m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax, color='red', linewidth=1.5, label='Células Selecionadas')

# Pontos ASC/DESC
points_sel.plot(ax=ax, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Pontos ORTHO
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Basemap e título
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery)
ax.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax.set_axis_off()
ax.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()
